# Data Quality Assessment

In [1]:
import pandas as pd
from utils.utils import load_data

In [2]:
tables = load_data()

The data was sucessfully loaded


In [5]:
#include table names to not have to look up the file
for table in tables.keys():
    print(table)

orders
customers
geolocation
order_items
reviews
payments
resellers
products


### Steps to analyze data quality per table
1. Missing values: count, percentage
2. Duplicates: rows and primary key ids
3. Validate foreign key
4. Impossible/Invalid values (example: negative freight)
5. Consistency (examples: orders without payment, or completed orders without delivery date)

## Orders

In [11]:
df_orders = tables["orders"]

#print first 2 rows of the table
df_orders.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [19]:
#number of rows in the table
print(f"The table has {df_orders.shape[0]} rows.")
print(f"The table has {df_orders.shape[1]} columns which are: {df_orders.columns.to_list()}.")

The table has 99441 rows.
The table has 8 columns which are: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'].


### Missing values

In [27]:
missing = pd.DataFrame({
    "missing_count": df_orders.isna().sum(),
    "missing_pct": (df_orders.isna().mean() * 100).round(2)
})

print(missing)

                               missing_count  missing_pct
order_id                                   0         0.00
customer_id                                0         0.00
order_status                               0         0.00
order_purchase_timestamp                   0         0.00
order_approved_at                        160         0.16
order_delivered_carrier_date            1783         1.79
order_delivered_customer_date           2965         2.98
order_estimated_delivery_date              0         0.00


The missing values are all related to dates, so it could just be a delivery status situation and not actual errors/missing information. Let's take a look at that:

In [38]:
#check delivery status for each column with null values
df_missing = df_orders[df_orders.isna().any(axis=1)]

,order_status,order_approved_at
6,invoiced,2017-04-13 13:25:17
44,shipped,2018-06-05 04:31:18
103,invoiced,2018-08-07 06:15:14
128,processing,2017-09-03 14:30:09
154,shipped,2018-01-11 02:32:30
...,...,...
99283,canceled,NaN
99313,processing,2018-01-09 07:18:05
99347,canceled,NaN
99348,unavailable,2017-08-28 15:44:47


### Duplicates

In [42]:
duplicated_rows = df_orders.duplicated().sum()

print(f"There are {duplicated_rows} duplicated rows.")

There are 0 duplicated rows.


In [43]:
duplicated_primary_key = df_orders['order_id'].duplicated().sum()

print(f"There are {duplicated_primary_key} rows with duplicated primary key")

There are 0 rows with duplicated primary key
